# FUNGI v4.1 — Pruning Pipeline
**Functional Unravelling of Network Geometry for Inference**

This notebook implements the complete FUNGI pruning pipeline: from dense-graph
ingestion through adaptive thresholding, DASH-kernel hyperparameter search,
Pareto triage, and final audit-cohort extraction.

All user-configurable paths and parameters are loaded from `fungi_config.yaml`.
Edit that file before running this notebook.

## Phase 1 — Environment Setup

### 1.1  Load Configuration

In [ ]:
import os
import yaml
from pathlib import Path

# Load pipeline configuration
CONFIG_PATH = Path("fungi_config.yaml")
with open(CONFIG_PATH) as fh:
    cfg = yaml.safe_load(fh)

# Enforce single-threaded BLAS before any numerical import
if cfg["runtime"].get("single_threaded_blas", True):
    for var in [
        "OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS",
        "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS",
        "NUMEXPR_NUM_THREADS",
    ]:
        os.environ[var] = "1"

# --- Resolve directory structure ----------------------------------------
PROJECT_ROOT = Path.cwd()
DATA_ROOT    = PROJECT_ROOT / "data"
INPUT_ROOT   = DATA_ROOT / "input"
OUTPUT_ROOT  = Path(cfg["output"]["root_dir"])
SRC_ROOT     = PROJECT_ROOT / "src"
FIGURES_ROOT = Path(cfg["output"]["figures_dir"])

# --- Resolve input file paths ------------------------------------------
RAW_GRAPH_PATH = Path(cfg["input"]["graph_path"])
SC_DATA_PATH   = Path(cfg["input"]["sc_data_path"]) if cfg["input"].get("sc_data_path") else None

# --- Create output directories -----------------------------------------
for phase_name in cfg["output"]["phases"]:
    (OUTPUT_ROOT / phase_name).mkdir(parents=True, exist_ok=True)
FIGURES_ROOT.mkdir(parents=True, exist_ok=True)

# --- Summary ------------------------------------------------------------
print("=" * 72)
print("FUNGI v3: Functional Unravelling of Network Geometry for Inference")
print("=" * 72)
print(f"  Project Root:     {PROJECT_ROOT}")
print(f"  Source Modules:   {SRC_ROOT}")
print(f"  Input Graph:      {RAW_GRAPH_PATH}")
print(f"  Input SC Data:    {SC_DATA_PATH}")
print(f"  Output Root:      {OUTPUT_ROOT}")
print(f"  Figures:          {FIGURES_ROOT}")
print("  Output directories created.")
print("=" * 72)

### 1.2  Library Imports

In [ ]:
import gc
import sys
import numpy as np
import pandas as pd
import scipy.sparse as sp
import networkx as nx
import scanpy as sc
import anndata as ad

# Add src/ to the import path
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print("Core libraries imported.")
print(f"src/ directory ({SRC_ROOT.name}) added to system path.")

### 1.3  Graph Ingestion

In [ ]:
from graph_utils import load_graph

raw_G, raw_sparse_mat = load_graph(RAW_GRAPH_PATH)

N_GENES = raw_sparse_mat.shape[0]
print(f"Graph loaded: {N_GENES:,} nodes, {raw_sparse_mat.nnz:,} edges")
print(f"Starting density: {nx.density(raw_G):.4%}")

## Phase 2 — Normalization

### 2.1  Adaptive Thresholding & Rank Standardization

In [ ]:
from scipy.stats import rankdata
from filtering import adaptive_threshold_filter

target_density = cfg["prefilter"]["target_density"]

# --- Adaptive threshold to reduce to the candidate pool -----------------
G_work = adaptive_threshold_filter(raw_sparse_mat, target_density=target_density)
gc.collect()
print(f"Candidate pool established at ~{target_density:.0%} density.")

# --- Rank-standardize edge weights (uniform [0, 1]) ---------------------
raw_weights = G_work.data
n_edges = len(raw_weights)
W_norm = rankdata(raw_weights, method="average") / n_edges
G_work.data = W_norm

print(f"W_norm range: [{W_norm.min():.6f}, {W_norm.max():.6f}]  ({n_edges:,} edges)")

# --- Extract raw out-degrees -------------------------------------------
raw_out_degrees = G_work.getnnz(axis=1)
print(f"Max raw out-degree: {raw_out_degrees.max():.0f}")

# --- Assemble candidate DataFrame --------------------------------------
G_coo = G_work.tocoo()
candidate_df = pd.DataFrame({
    "source": G_coo.row,
    "target": G_coo.col,
    "weight": W_norm,
    "degree": raw_out_degrees[G_coo.row],
})

export_path = OUTPUT_ROOT / "phase2_normalization" / "candidate_edges_normalized.parquet"
export_path.parent.mkdir(parents=True, exist_ok=True)
candidate_df.to_parquet(export_path, index=False)

print(f"Exported {len(candidate_df):,} candidate edges to {export_path.name}")
display(candidate_df.head())

del G_work, G_coo
gc.collect()

### 2.2  Normalization Diagnostic Report

In [ ]:
total_possible = N_GENES * N_GENES
original_edges = raw_sparse_mat.nnz
work_edges     = len(candidate_df)

report = f"""
{'=' * 72}
FUNGI v4 — Phase 2: Normalization Report
{'=' * 72}
  Nodes (genes):         {N_GENES:,}
  Possible edges:        {total_possible:,}

  Original dense graph
    Edges:               {original_edges:,}
    Density:             {(original_edges / total_possible) * 100:.4f}%
    Mean degree:         {original_edges / N_GENES:.2f}

  Candidate pool
    Edges:               {work_edges:,}
    Density:             {(work_edges / total_possible) * 100:.4f}%
    Mean degree:         {work_edges / N_GENES:.2f}
    Edges removed:       {original_edges - work_edges:,} ({((original_edges - work_edges) / original_edges) * 100:.2f}%)

  Engine-ready metrics
    Weight range:        [{candidate_df['weight'].min():.6f}, {candidate_df['weight'].max():.6f}]
    Max raw degree:      {candidate_df['degree'].max():.0f}
{'=' * 72}
"""
print(report)

## Phase 3 — DASH Search Engine

### 3.1  Global Search — Latin Hypercube Generation

In [ ]:
import importlib
import src.ml_search
importlib.reload(src.ml_search)
from src.ml_search import generate_latin_hypercube

# --- Extract perturbation targets for local-connectivity guardrail ------
perturbed_nodes = np.array([], dtype=int)

if SC_DATA_PATH is not None:
    pert_col   = cfg["input"]["perturbation_column"]
    ctrl_label = cfg["input"]["control_label"]

    adata = sc.read_h5ad(SC_DATA_PATH, backed="r")
    conditions = adata.obs[pert_col].unique()
    perturbed_gene_names = [g for g in conditions if g != ctrl_label]

    npz_data = np.load(RAW_GRAPH_PATH, allow_pickle=True)

    # Resolve gene labels: check common key names
    _gene_keys = ["genes", "gene_names", "node_names", "labels"]
    original_genes = None
    for gk in _gene_keys:
        if gk in npz_data.files:
            original_genes = npz_data[gk]
            break

    if original_genes is None:
        raise KeyError(
            f"No gene-label array found in {RAW_GRAPH_PATH.name}. "
            f"Available keys: {npz_data.files}"
        )

    gene_to_idx = {gene: idx for idx, gene in enumerate(original_genes)}
    perturbed_nodes = np.array(
        [gene_to_idx[g] for g in perturbed_gene_names if g in gene_to_idx]
    )
    print(f"Perturbation guardrail active: {len(perturbed_nodes)} targets.")
else:
    print("No single-cell data provided; perturbation guardrail disabled.")

# --- Extract arrays for the engine -------------------------------------
sources_arr = candidate_df["source"].values
targets_arr = candidate_df["target"].values
W_arr       = candidate_df["weight"].values
D_arr       = candidate_df["degree"].values

# --- Generate Latin Hypercube sample points ----------------------------
n_global = cfg["global_search"]["n_samples"]
global_params = generate_latin_hypercube(n_genes=N_GENES, n_samples=n_global)

print(f"Prepared {len(W_arr):,} candidate edges as engine-ready arrays.")
print(f"Generated {len(global_params):,} global search coordinates (LHS).")


### 3.1a  Global Search — Hyperparameter Distributions

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

hp_names = ["beta", "gamma", "delta", "kappa", "k_core", "lambda"]
df_global_viz = pd.DataFrame(global_params, columns=hp_names)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("Phase 3.1: Latin Hypercube Distributions (Global Search)", fontsize=16)

for ax, col in zip(axes.flatten(), df_global_viz.columns):
    sns.histplot(df_global_viz[col], bins=50, ax=ax, color="teal", kde=False)
    ax.set_title(col)
    ax.set_ylabel("Count")

plt.tight_layout()
plt.show()

### 3.2  Global Search — Batch Execution

In [ ]:
import time
import shutil
import joblib
import contextlib
from tqdm.notebook import tqdm

import src.engine
importlib.reload(src.engine)
from src.engine import execute_dash_batch

phase3_dir = OUTPUT_ROOT / "phase3_search_engine"
shard_dir  = phase3_dir / "temp_shards"
phase3_dir.mkdir(parents=True, exist_ok=True)
shard_dir.mkdir(parents=True, exist_ok=True)

global_save_path = phase3_dir / "global_search_results.csv"

CHUNK_SIZE = cfg["global_search"]["chunk_size"]
N_JOBS     = cfg["global_search"]["n_jobs"]
TOTAL_RUNS = len(global_params)
chunks = [global_params[i:i + CHUNK_SIZE] for i in range(0, TOTAL_RUNS, CHUNK_SIZE)]


@contextlib.contextmanager
def tqdm_joblib(tqdm_object):
    """Context manager to patch joblib with a tqdm progress bar."""
    class _Callback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = _Callback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old


print(f"Executing global search: {TOTAL_RUNS:,} runs across {len(chunks)} shards ...")
t0 = time.time()
all_results = []

for idx, chunk in enumerate(chunks):
    shard_file = shard_dir / f"shard_{idx:03d}.parquet"

    if shard_file.exists():
        print(f"  Shard {idx + 1}/{len(chunks)} found on disk — skipping.")
        all_results.append(pd.read_parquet(shard_file))
        continue

    print(f"  Computing shard {idx + 1}/{len(chunks)} ({len(chunk):,} runs) ...")
    with tqdm_joblib(tqdm(desc=f"Shard {idx + 1}", total=len(chunk))):
        df_shard = execute_dash_batch(
            param_list=chunk,
            W_arr=W_arr,
            D_arr=D_arr,
            sources_arr=sources_arr,
            targets_arr=targets_arr,
            n_genes=N_GENES,
            perturbed_nodes=perturbed_nodes,
            n_jobs=N_JOBS,
        )
    df_shard.to_parquet(shard_file, index=False)
    all_results.append(df_shard)

elapsed = (time.time() - t0) / 60

# --- Compile and verify -------------------------------------------------
df_global = pd.concat(all_results, ignore_index=True)

if len(df_global) == TOTAL_RUNS:
    print(f"Integrity check passed: {TOTAL_RUNS:,} runs recovered.")
    df_global.to_csv(global_save_path, index=False)
    shutil.rmtree(shard_dir)
    print(f"Results saved to {global_save_path.name}; temporary shards removed.")
else:
    print(f"WARNING: Expected {TOTAL_RUNS:,} rows, got {len(df_global):,}. Shards preserved for debugging.")

shattered = df_global["is_shattered"].sum()
best_loss = df_global.loc[df_global["is_shattered"] == 0, "utopia_loss"].min()

print(f"\nGlobal search complete in {elapsed:.2f} min.")
print(f"  Shattered: {shattered:,} ({shattered / len(df_global) * 100:.1f}%)")
print(f"  Best loss: {best_loss:.4f}")

### 3.3  Global Search — Diagnostic Report

In [ ]:
total_possible = N_GENES * N_GENES
df_viable = df_global[df_global["is_shattered"] == 0].copy()
df_shattered = df_global[df_global["is_shattered"] == 1]

if len(df_viable) == 0:
    print("WARNING: All graphs shattered. Review utopian thresholds.")
else:
    lam = df_viable["lambda"].values
    edges = lam * N_GENES
    density = (edges / total_possible) * 100

    best_idx   = df_viable["utopia_loss"].idxmin()
    best_q_idx = df_viable["Q"].idxmax()
    best_g_idx = (df_viable["Gini"] - 0.75).abs().idxmin()
    best_s_idx = (df_viable["S_max"] - 0.05).abs().idxmin()

    b = df_viable.loc[best_idx]
    bq = df_viable.loc[best_q_idx]
    bg = df_viable.loc[best_g_idx]
    bs = df_viable.loc[best_s_idx]

    print("=" * 65)
    print("FUNGI — Phase 3 Global Search Report")
    print("=" * 65)
    print(f"Total runs:        {len(df_global):,}")
    print(f"Viable graphs:     {len(df_viable):,} ({len(df_viable) / len(df_global) * 100:.1f}%)")
    print(f"Shattered graphs:  {len(df_shattered):,}\n")

    print("Lambda (avg degree)")
    print(f"  Min: {lam.min():.2f}   Mean: {lam.mean():.2f}   Max: {lam.max():.2f}\n")

    print("Edge count")
    print(f"  Min: {int(edges.min()):,}   Mean: {int(edges.mean()):,}   Max: {int(edges.max()):,}\n")

    print("Global density")
    print(f"  Min: {density.min():.4f}%   Mean: {density.mean():.4f}%   Max: {density.max():.4f}%\n")

    if "n_orphans" in df_viable.columns:
        print(f"Orphan genes (avg): {df_viable['n_orphans'].mean():.0f} / {N_GENES}")
        print(f"Islands (avg):      {df_viable['n_islands'].mean():.1f}\n")

    print("Topology averages")
    print(f"  Q:    {df_viable['Q'].mean():.4f}")
    print(f"  Gini: {df_viable['Gini'].mean():.4f}")
    print(f"  S_max:{df_viable['S_max'].mean() * 100:.2f}%\n")

    print("Metric leaders")
    print(f"  Best overall  — Loss: {b['utopia_loss']:.4f}  Q: {b['Q']:.4f}  Gini: {b['Gini']:.4f}  S_max: {b['S_max']*100:.2f}%")
    print(f"  Best Q        — Loss: {bq['utopia_loss']:.4f}  Q: {bq['Q']:.4f}  Gini: {bq['Gini']:.4f}  S_max: {bq['S_max']*100:.2f}%")
    print(f"  Best Gini     — Loss: {bg['utopia_loss']:.4f}  Q: {bg['Q']:.4f}  Gini: {bg['Gini']:.4f}  S_max: {bg['S_max']*100:.2f}%")
    print(f"  Best S_max    — Loss: {bs['utopia_loss']:.4f}  Q: {bs['Q']:.4f}  Gini: {bs['Gini']:.4f}  S_max: {bs['S_max']*100:.2f}%")
    print("=" * 65)

### 3.4  Global Search — 3D Hyperparameter Landscape

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

df_plot = df_global[df_global["is_shattered"] == 0].copy()
df_plot["loss_pct"] = df_plot["utopia_loss"].rank(pct=True, ascending=False)

combos = [
    ("beta", "gamma", "lambda"),
    ("beta", "delta", "gamma"),
    ("gamma", "kappa", "lambda"),
    ("delta", "k_core", "lambda"),
    ("beta", "kappa", "delta"),
    ("k_core", "gamma", "beta"),
    ("lambda", "delta", "kappa"),
    ("beta", "lambda", "k_core"),
    ("gamma", "delta", "kappa"),
]

fig = plt.figure(figsize=(24, 24))
for i, (x, y, z) in enumerate(combos, 1):
    ax = fig.add_subplot(3, 3, i, projection="3d")
    sc = ax.scatter(
        df_plot[x], df_plot[y], df_plot[z],
        c=df_plot["loss_pct"], cmap="inferno", s=15, alpha=0.8,
        vmin=0.0, vmax=1.0,
    )
    ax.set_xlabel(x, labelpad=10)
    ax.set_ylabel(y, labelpad=10)
    ax.set_zlabel(z, labelpad=10)
    ax.set_title(f"{x}  vs  {y}  vs  {z}", fontweight="bold", pad=15)
    ax.view_init(elev=30, azim=45)

cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
cbar = fig.colorbar(sc, cax=cbar_ax)
cbar.set_label(
    "Utopian-Loss Percentile (brighter = better)",
    rotation=270, labelpad=25, fontsize=14, fontweight="bold",
)
plt.subplots_adjust(right=0.88, wspace=0.1, hspace=0.2)
plt.show()

### 3.5  Global Search — 2D Fitness Density

In [ ]:
import seaborn as sns
import matplotlib.gridspec as gridspec
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

df_kde = df_global[df_global["is_shattered"] == 0].copy()
max_loss = df_kde["utopia_loss"].max()
df_kde["fitness_weight"] = (max_loss - df_kde["utopia_loss"]) ** 2

plot_pairs = [
    ("lambda", "gamma",  "Global Budget (\u03BB)",  "Hub Penalty (\u03B3)"),
    ("lambda", "delta",  "Global Budget (\u03BB)",  "Motif Reward (\u03B4)"),
    ("lambda", "kappa",  "Global Budget (\u03BB)",  "Saturation Limit (\u03BA)"),
    ("gamma",  "delta",  "Hub Penalty (\u03B3)",    "Motif Reward (\u03B4)"),
    ("gamma",  "kappa",  "Hub Penalty (\u03B3)",    "Saturation Limit (\u03BA)"),
    ("delta",  "k_core", "Motif Reward (\u03B4)",   "Core Stringency (k_core)"),
    ("beta",   "gamma",  "Signal Trust (\u03B2)",   "Hub Penalty (\u03B3)"),
    ("beta",   "delta",  "Signal Trust (\u03B2)",   "Motif Reward (\u03B4)"),
    ("beta",   "lambda", "Signal Trust (\u03B2)",   "Global Budget (\u03BB)"),
]

sns.set_theme(style="white", context="paper", font_scale=1.3)
fig = plt.figure(figsize=(18, 18))
outer = gridspec.GridSpec(
    3, 3, wspace=0.35, hspace=0.25, top=0.92, right=0.96, left=0.06, bottom=0.12,
)

for i, (xc, yc, xl, yl) in enumerate(plot_pairs):
    inner = gridspec.GridSpecFromSubplotSpec(
        2, 2, subplot_spec=outer[i], wspace=0.05, hspace=0.05,
        width_ratios=[4, 1], height_ratios=[1, 4],
    )
    ax_main  = fig.add_subplot(inner[1, 0])
    ax_top   = fig.add_subplot(inner[0, 0], sharex=ax_main)
    ax_right = fig.add_subplot(inner[1, 1], sharey=ax_main)

    xmin, xmax = df_global[xc].min(), df_global[xc].max()
    ymin, ymax = df_global[yc].min(), df_global[yc].max()

    sns.kdeplot(data=df_kde, x=xc, y=yc, weights="fitness_weight",
                ax=ax_main, cmap="rocket", fill=True, thresh=0.01, levels=20, alpha=0.9,
                clip=((xmin, xmax), (ymin, ymax)))
    sns.kdeplot(data=df_kde, x=xc, y=yc, weights="fitness_weight",
                ax=ax_main, color="black", linewidths=0.5, thresh=0.01, levels=15, alpha=0.6,
                clip=((xmin, xmax), (ymin, ymax)))

    sns.kdeplot(data=df_kde, x=xc, weights="fitness_weight",
                ax=ax_top, color="#f97150", fill=True, alpha=0.8, linewidth=1.5,
                clip=(xmin, xmax))
    sns.kdeplot(data=df_kde, y=yc, weights="fitness_weight",
                ax=ax_right, color="#f97150", fill=True, alpha=0.8, linewidth=1.5,
                clip=(ymin, ymax))

    ax_top.axis("off")
    ax_right.axis("off")
    ax_main.set_xlim(xmin, xmax)
    ax_main.set_ylim(ymin, ymax)
    ax_main.set_xlabel(xl, weight="bold")
    ax_main.set_ylabel(yl, weight="bold")
    ax_top.set_title(f"{xc} vs {yc}", weight="bold", fontsize=15, pad=15)

sm = ScalarMappable(cmap="rocket", norm=Normalize(vmin=0, vmax=1))
sm.set_array([])
cax = fig.add_axes([0.1, 0.04, 0.8, 0.015])
cb = fig.colorbar(sm, cax=cax, orientation="horizontal")
min_l = df_kde["utopia_loss"].min()
max_l = df_kde["utopia_loss"].max()
cb.set_ticks([0, 0.5, 1])
cb.set_ticklabels([
    f"High loss ({max_l:.3f})",
    f"Moderate ({(min_l + max_l) / 2:.3f})",
    f"Optimal ({min_l:.3f})",
], fontsize=12)
cb.set_label("Fitness Density (weighted by utopian loss)", weight="bold", fontsize=16, labelpad=10)

fig.suptitle("Global Search: Viable Hyperparameter Landscape", y=0.98, weight="bold", fontsize=24)
plt.savefig(FIGURES_ROOT / "global_search_2d_fitness.png", dpi=300, bbox_inches="tight")
plt.show()

### 3.6  Targeted Search — Surrogate Training & Candidate Generation

In [ ]:
import src.ml_search
importlib.reload(src.ml_search)
from src.ml_search import (
    train_tri_model_surrogate,
    generate_smart_targeted_coordinates,
    get_agnostic_bounds,
)

ts_cfg = cfg["targeted_search"]
feature_cols = ["beta", "gamma", "delta", "kappa", "k_core", "lambda"]

# Re-read global ledger from disk for a clean state
df_ledger = pd.read_csv(global_save_path)

lower_bounds, upper_bounds = get_agnostic_bounds(N_GENES)

gmm, rf_clf, rf_reg = train_tri_model_surrogate(
    df_ledger=df_ledger,
    feature_cols=feature_cols,
    target_col="utopia_loss",
    elite_percentile=ts_cfg["elite_percentile"],
)

targeted_params = generate_smart_targeted_coordinates(
    gmm=gmm,
    rf_clf=rf_clf,
    rf_reg=rf_reg,
    lower_bounds=lower_bounds,
    upper_bounds=upper_bounds,
    n_hallucinate=ts_cfg["n_hallucinate"],
    n_needed=ts_cfg["n_select"],
    min_survival_prob=ts_cfg["min_survival_prob"],
)

print(f"Surrogate trained. Generated {len(targeted_params):,} targeted search coordinates.")

# --- Overlay comparison plot --------------------------------------------
df_targeted_viz = pd.DataFrame(targeted_params, columns=feature_cols)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("Phase 3.6: Targeted vs. Global Hyperparameter Focus", fontsize=16)

for ax, col in zip(axes.flatten(), feature_cols):
    sns.kdeplot(df_ledger[col], ax=ax, color="lightgrey", fill=True, label="Global (LHS)")
    sns.kdeplot(df_targeted_viz[col], ax=ax, color="purple", fill=True, label="Targeted")
    ax.set_title(col)
    ax.set_ylabel("Density")
    if ax == axes.flatten()[0]:
        ax.legend()

plt.tight_layout()
plt.show()

### 3.7  Targeted Search — Batch Execution

In [ ]:
print(f"Executing targeted search: {len(targeted_params):,} runs on {ts_cfg['n_jobs']} workers ...")
t0 = time.time()

with tqdm_joblib(tqdm(desc="Targeted Search", total=len(targeted_params))):
    df_targeted = execute_dash_batch(
        param_list=targeted_params,
        W_arr=W_arr,
        D_arr=D_arr,
        sources_arr=sources_arr,
        targets_arr=targets_arr,
        n_genes=N_GENES,
        perturbed_nodes=perturbed_nodes,
        n_jobs=ts_cfg["n_jobs"],
    )

elapsed = (time.time() - t0) / 60

targeted_save_path = phase3_dir / "targeted_search_results.csv"
df_targeted.to_csv(targeted_save_path, index=False)

shattered = df_targeted["is_shattered"].sum()
best_loss = df_targeted.loc[df_targeted["is_shattered"] == 0, "utopia_loss"].min()

print(f"\nTargeted search complete in {elapsed:.2f} min.")
print(f"  Total runs:  {len(df_targeted):,}")
print(f"  Shattered:   {shattered:,} ({shattered / len(df_targeted) * 100:.1f}%)")
print(f"  Best loss:   {best_loss:.4f}")
print(f"  Saved to:    {targeted_save_path.name}")

### 3.8  Targeted Search — Diagnostic Visualizations

In [ ]:
# --- 3D Landscape -------------------------------------------------------
df_t_plot = df_targeted[df_targeted["is_shattered"] == 0].copy()
df_t_plot["loss_pct"] = df_t_plot["utopia_loss"].rank(pct=True, ascending=False)

fig = plt.figure(figsize=(24, 24))
for i, (x, y, z) in enumerate(combos, 1):
    ax = fig.add_subplot(3, 3, i, projection="3d")
    sc = ax.scatter(
        df_t_plot[x], df_t_plot[y], df_t_plot[z],
        c=df_t_plot["loss_pct"], cmap="inferno", s=15, alpha=0.8,
        vmin=0.0, vmax=1.0,
    )
    ax.set_xlabel(x, labelpad=10)
    ax.set_ylabel(y, labelpad=10)
    ax.set_zlabel(z, labelpad=10)
    ax.set_title(f"{x}  vs  {y}  vs  {z}", fontweight="bold", pad=15)
    ax.view_init(elev=30, azim=45)

cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
cbar = fig.colorbar(sc, cax=cbar_ax)
cbar.set_label(
    "Utopian-Loss Percentile (brighter = better)",
    rotation=270, labelpad=25, fontsize=14, fontweight="bold",
)
plt.suptitle("Targeted Search: 3D Hyperparameter Landscape", y=0.95, weight="bold", fontsize=28)
plt.subplots_adjust(right=0.88, wspace=0.1, hspace=0.2)
plt.savefig(FIGURES_ROOT / "targeted_search_3d_landscape.png", dpi=300, bbox_inches="tight")
plt.show()

### 3.9  Targeted Search — Tabular Analysis

In [ ]:
df_t_viable = df_targeted[df_targeted["is_shattered"] == 0].copy()
top_5_cutoff = df_t_viable["utopia_loss"].quantile(0.05)
df_t_elite = df_t_viable[df_t_viable["utopia_loss"] <= top_5_cutoff]

params = ["beta", "gamma", "delta", "kappa", "k_core", "lambda"]
correlations = (
    df_t_viable[params + ["utopia_loss"]]
    .corr(method="spearman")["utopia_loss"]
    .drop("utopia_loss")
)

param_summary = df_t_elite[params].describe().T[["min", "25%", "50%", "mean", "75%", "max"]]
param_summary["loss_corr"] = correlations

topo_cols = ["utopia_loss", "Q", "Gini", "S_max", "n_islands", "n_orphans"]
topo_summary = df_t_elite[topo_cols].describe().T[["min", "25%", "50%", "mean", "75%", "max"]]
topo_summary.loc["S_max"] *= 100
topo_summary.rename(index={"S_max": "S_max (%)"}, inplace=True)

print("=" * 72)
print("Targeted Search — Top 5% Elite Analysis")
print("=" * 72)
print(f"Total runs:       {len(df_targeted):,}")
print(f"Viable runs:      {len(df_t_viable):,}")
print(f"Elite runs (5%):  {len(df_t_elite):,}")
print(f"Best loss:        {df_t_elite['utopia_loss'].min():.4f}")
print(f"Cutoff loss:      {top_5_cutoff:.4f}")
print("-" * 72)
print("Hyperparameter distribution (elite zone):")
print(param_summary.round(4).to_string())
print("\n" + "-" * 72)
print("Topological readouts (elite zone):")
print(topo_summary.round(4).to_string())
print("=" * 72)

### 3.10  Master Merge & Top-K Extraction

In [ ]:
top_k = cfg["pareto"]["top_k_generalists"]

global_save_path = phase3_dir / "global_search_results.csv"

df_all = pd.concat(
    [pd.read_csv(global_save_path), pd.read_csv(targeted_save_path)],
    ignore_index=True,
)

df_all_viable = df_all[df_all["is_shattered"] == 0].copy()
df_all_viable = df_all_viable.drop_duplicates(subset=params, keep="first")

df_topk = df_all_viable.sort_values("utopia_loss").head(top_k).copy()
topk_path = phase3_dir / f"master_top{top_k}.csv"
df_topk.to_csv(topk_path, index=False)

topo_cols = ["utopia_loss", "Q", "Gini", "S_max", "n_islands", "n_orphans"]

print("=" * 65)
print(f"Master Merge — Top {top_k} Generalist Cohort")
print("=" * 65)
print(f"Total unique viable:  {len(df_all_viable):,}")
print(f"Selected top-{top_k}:     {len(df_topk):,}")
print(f"Best loss:            {df_topk['utopia_loss'].min():.4f}")
print(f"Worst loss:           {df_topk['utopia_loss'].max():.4f}")
print(f"Saved to:             {topk_path.name}\n")
print("Topological averages:")
print(df_topk[topo_cols].mean().to_frame(name="Mean").round(4))
print("=" * 65)

## Phase 4 — Pareto Triage

### 4.1  Distance to Utopia — Dense Baseline vs. Top-K

In [ ]:
import powerlaw
import igraph as ig
import warnings
from graph_utils import calculate_gini

warnings.filterwarnings("ignore", category=RuntimeWarning)

# --- Top-K averages for the six output parameters -----------------------
top500_means = {
    "alpha": df_topk["alpha"].mean(),
    "rho":   df_topk["rho"].mean(),
    "Gini":  df_topk["Gini"].mean(),
    "C":     df_topk["C"].mean(),
    "Q":     df_topk["Q"].mean(),
    "S_max": df_topk["S_max"].mean(),
}

# --- Reconstruct dense graph for baseline comparison --------------------
print("Computing dense-graph structural baseline ...")
with np.load(RAW_GRAPH_PATH, allow_pickle=True) as data:
    matrix_key = [k for k in data.files if k != "genes"][0]
    A_dense_raw = data[matrix_key]

A_dense = sp.csr_matrix(A_dense_raw)[:N_GENES, :N_GENES]

# Build igraph directly from sparse matrix (C backend, much faster than NetworkX)
A_dense_coo = sp.coo_matrix(A_dense)
edges = list(zip(A_dense_coo.row.tolist(), A_dense_coo.col.tolist()))
ig_dense = ig.Graph(n=N_GENES, edges=edges, directed=True,
                    edge_attrs={'weight': A_dense_coo.data.tolist()})

degrees = [d for d in ig_dense.degree()]
fit = powerlaw.Fit(degrees, discrete=True, verbose=False)

ig_undirected = ig_dense.as_undirected(mode="collapse", combine_edges=dict(weight="sum"))

dense_metrics = {
    "alpha": fit.power_law.alpha,
    "rho":   ig_dense.assortativity_degree(directed=True),
    "Gini":  calculate_gini(degrees),
    "C":     ig_undirected.transitivity_undirected(),
    "Q":     ig_undirected.community_multilevel().modularity,
    "S_max": max(degrees) / N_GENES,
}

# --- Compute per-parameter distance to utopian zone --------------------
utopia = cfg["pareto"]["utopian_bounds"]
key_map = {"alpha": "alpha", "rho": "rho", "gini": "Gini", "C": "C", "Q": "Q", "S_max": "S_max"}

def dist_to_box(val, lo, hi):
    if val < lo: return lo - val
    if val > hi: return val - hi
    return 0.0

rows = []
for cfg_key, bounds in utopia.items():
    m = key_map[cfg_key]
    d_dist = dist_to_box(dense_metrics[m], bounds[0], bounds[1])
    t_dist = dist_to_box(top500_means[m], bounds[0], bounds[1])
    rows.append({
        "Parameter": m,
        "Utopian Zone": f"[{bounds[0]}, {bounds[1]}]",
        "Dense": round(dense_metrics[m], 4),
        "Dense Dist.": round(d_dist, 4),
        f"Top-{top_k} Avg": round(top500_means[m], 4),
        f"Top-{top_k} Dist.": round(t_dist, 4),
        "Improvement": round(d_dist - t_dist, 4),
    })

df_compare = pd.DataFrame(rows).sort_values("Improvement", ascending=False)

print("\n" + "=" * 100)
print(f"Distance to Utopia: Dense Baseline vs. Top-{top_k} Average")
print("=" * 100)
print(df_compare.to_string(index=False))
print("=" * 100)


### 4.2  Archetypal Scoring & Audit-Cohort Extraction

In [ ]:
phase4_dir = OUTPUT_ROOT / "phase4_pareto_triage"
phase4_dir.mkdir(parents=True, exist_ok=True)

df = df_topk.copy()

# --- Compute raw penalties per output metric ----------------------------
raw_alpha = (df["alpha"] - 2.3) ** 2
raw_C     = ((df["C"] - 0.35) / 0.35) ** 2
raw_Q     = np.where(df["Q"] < 0.40, ((0.40 - df["Q"]) / 0.40) ** 2, 0.0)
raw_G_lo  = np.where(df["Gini"] < 0.65, ((0.65 - df["Gini"]) / 0.65) ** 2, 0.0)
raw_G_hi  = np.where(df["Gini"] > 0.85, ((df["Gini"] - 0.85) / 0.85) ** 2, 0.0)
raw_G     = raw_G_lo + raw_G_hi
raw_rho   = (df["rho"] - (-0.3)) ** 2
raw_sat   = np.maximum(0.0, (df["S_max"] - df["kappa"]) / df["kappa"]) ** 2

# --- Score under each archetype ----------------------------------------
arch_weights = cfg["pareto"]["archetype_weights"]

for name, w in arch_weights.items():
    loss_sq = (
        w["alpha"] * raw_alpha
        + w["C"]     * raw_C
        + w["Q"]     * raw_Q
        + w["gini"]  * raw_G
        + w["rho"]   * raw_rho
        + w["sat"]   * raw_sat
    )
    df[f"loss_{name}"]  = np.sqrt(loss_sq)
    df[f"rank_{name}"]  = df[f"loss_{name}"].rank(method="min")

# --- Borda-style consensus rank ----------------------------------------
rank_cols = [f"rank_{n}" for n in arch_weights]
df["mean_rank"]      = df[rank_cols].mean(axis=1)
df["rank_consensus"] = df["mean_rank"].rank(method="min")

# --- Greedy unique selection -------------------------------------------
n_per = cfg["pareto"]["graphs_per_archetype"]
categories = list(arch_weights.keys()) + ["consensus"]

cohort = []
claimed = set()

for cat in categories:
    for idx, row in df.sort_values(f"rank_{cat}").iterrows():
        if idx not in claimed:
            r = row.copy()
            r["archetype"] = cat.capitalize()
            cohort.append(r)
            claimed.add(idx)
        if len([c for c in cohort if c["archetype"] == cat.capitalize()]) >= n_per:
            break

df_audit = pd.DataFrame(cohort)
audit_path = phase4_dir / "audit_cohort.csv"
df_audit.to_csv(audit_path, index=False)

metrics_show = ["alpha", "C", "Q", "Gini", "rho", "S_max", "n_islands"]
breakdown = df_audit.groupby("archetype")[metrics_show].mean()
breakdown["S_max"] *= 100
breakdown.rename(columns={"S_max": "S_max (%)"}, inplace=True)

print("=" * 70)
print(f"Phase 4 — {len(df_audit)}-Graph Audit Cohort Extracted")
print("=" * 70)
print(f"Saved to: {audit_path.name}\n")
print("Cohort breakdown (archetype averages):")
print(breakdown.round(4).to_string())
print("=" * 70)

### 4.3  Utopian Convergence — 2D Density with Target Zones

In [ ]:
import matplotlib.patches as patches

df_p4 = df_topk.copy()
max_loss = df_p4["utopia_loss"].max()
df_p4["fitness_weight"] = (max_loss - df_p4["utopia_loss"]) ** 2

plot_pairs_p4 = [
    ("alpha", "rho",   "Scale-Free Tail (\u03B1)",  "Assortativity (\u03C1)",      (2.1, 2.5),   (-0.40, -0.20)),
    ("alpha", "S_max", "Scale-Free Tail (\u03B1)",  "Hub Saturation (S_max)",   (2.1, 2.5),   (0.04, 0.10)),
    ("rho",   "S_max", "Assortativity (\u03C1)",    "Hub Saturation (S_max)",   (-0.40, -0.20), (0.04, 0.10)),
    ("Gini",  "S_max", "Inequality (Gini)",      "Hub Saturation (S_max)",   (0.65, 0.85), (0.04, 0.10)),
    ("alpha", "Gini",  "Scale-Free Tail (\u03B1)",  "Inequality (Gini)",        (2.1, 2.5),   (0.65, 0.85)),
    ("rho",   "Gini",  "Assortativity (\u03C1)",    "Inequality (Gini)",        (-0.40, -0.20), (0.65, 0.85)),
    ("C",     "Q",     "Clustering (C)",         "Modularity (Q)",           (0.30, 0.40), (0.35, 0.45)),
    ("C",     "S_max", "Clustering (C)",         "Hub Saturation (S_max)",   (0.30, 0.40), (0.04, 0.10)),
    ("Gini",  "Q",     "Inequality (Gini)",      "Modularity (Q)",           (0.65, 0.85), (0.35, 0.45)),
]

sns.set_theme(style="white", context="paper", font_scale=1.3)
fig = plt.figure(figsize=(18, 18))
outer = gridspec.GridSpec(
    3, 3, wspace=0.35, hspace=0.25, top=0.92, right=0.96, left=0.06, bottom=0.12,
)

for i, (xc, yc, xl, yl, xb, yb) in enumerate(plot_pairs_p4):
    inner = gridspec.GridSpecFromSubplotSpec(
        2, 2, subplot_spec=outer[i], wspace=0.05, hspace=0.05,
        width_ratios=[4, 1], height_ratios=[1, 4],
    )
    ax_main  = fig.add_subplot(inner[1, 0])
    ax_top   = fig.add_subplot(inner[0, 0], sharex=ax_main)
    ax_right = fig.add_subplot(inner[1, 1], sharey=ax_main)

    sns.kdeplot(data=df_p4, x=xc, y=yc, weights="fitness_weight",
                ax=ax_main, cmap="rocket", fill=True, thresh=0.01, levels=20, alpha=0.9)
    sns.kdeplot(data=df_p4, x=xc, y=yc, weights="fitness_weight",
                ax=ax_main, color="black", linewidths=0.5, thresh=0.01, levels=15, alpha=0.6)

    rect = patches.Rectangle(
        (xb[0], yb[0]), xb[1] - xb[0], yb[1] - yb[0],
        linewidth=2.5, edgecolor="#00ff00", facecolor="none",
        linestyle="--", zorder=10, alpha=0.9,
    )
    ax_main.add_patch(rect)

    sns.kdeplot(data=df_p4, x=xc, weights="fitness_weight",
                ax=ax_top, color="#f97150", fill=True, alpha=0.8, linewidth=1.5)
    sns.kdeplot(data=df_p4, y=yc, weights="fitness_weight",
                ax=ax_right, color="#f97150", fill=True, alpha=0.8, linewidth=1.5)

    ax_top.axis("off")
    ax_right.axis("off")
    ax_main.set_xlabel(xl, weight="bold")
    ax_main.set_ylabel(yl, weight="bold")
    ax_top.set_title(f"{xc} vs {yc}", weight="bold", fontsize=15, pad=15)

sm = ScalarMappable(cmap="rocket", norm=Normalize(vmin=0, vmax=1))
sm.set_array([])
cax = fig.add_axes([0.1, 0.04, 0.8, 0.015])
cb = fig.colorbar(sm, cax=cax, orientation="horizontal")
min_l = df_p4["utopia_loss"].min()
max_l = df_p4["utopia_loss"].max()
cb.set_ticks([0, 0.5, 1])
cb.set_ticklabels([
    f"Edge of Top-{top_k} ({max_l:.3f})",
    f"Strong ({(min_l + max_l) / 2:.3f})",
    f"Peak ({min_l:.3f})",
], fontsize=12)
cb.set_label("Fitness Density (weighted by utopian loss)", weight="bold", fontsize=16, labelpad=10)

fig.suptitle(
    f"Top-{top_k} Biological Parameters: Convergence on Utopian Targets",
    y=0.98, weight="bold", fontsize=24,
)
green_legend = patches.Patch(
    color="#00ff00", fill=False, linestyle="--", linewidth=2.5,
    label="Utopian Target Zone",
)
fig.legend(handles=[green_legend], loc="upper right", bbox_to_anchor=(0.95, 0.97), fontsize=14, frameon=True)

plt.savefig(FIGURES_ROOT / "pareto_utopia_convergence.png", dpi=300, bbox_inches="tight")
plt.show()

## Phase 5 — Export Pipeline Artifacts

In [ ]:
import joblib as jl

artifact_path = OUTPUT_ROOT / "phase2_normalization" / "pipeline_artifacts.joblib"

artifacts = {
    "candidate_df": candidate_df,
    "N_GENES": N_GENES,
}

jl.dump(artifacts, artifact_path)
print(f"Pipeline artifacts exported to {artifact_path.name}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Phase 6 — Cohort Rebuild & HYPHAE Manifest Export
# ═══════════════════════════════════════════════════════════════
#
# Reconstructs every champion graph from df_audit hyperparameters
# using the same logic as the audit notebook (Section 1.4):
#   compute_dynamic_topology → DASH omega → dual_pass_pruning → adjacency
#
# Saves each graph as {graph_id}_adjacency.npz and writes
# hyphae_manifest.csv for FUNGIv5_HYPHAE.ipynb.
#
# All arrays (W_arr, D_arr, sources_arr, targets_arr, N_GENES,
# perturbed_nodes) must already be in memory from earlier cells.
# ═══════════════════════════════════════════════════════════════

import scipy.sparse as sp
import importlib
import src.engine
importlib.reload(src.engine)
from src.engine import compute_dynamic_topology, dual_pass_pruning
from tqdm.notebook import tqdm

phase6_dir = OUTPUT_ROOT / "phase6_final_delivery"
phase6_dir.mkdir(parents=True, exist_ok=True)

# ─── Step 1: Assign stable graph_ids (mirrors audit notebook logic) ───
# Numbered within archetype by ascending utopia_loss.
# Generalist_01 = best Generalist, Generalist_02 = second best, etc.
df_manifest = df_audit.sort_values(["archetype", "utopia_loss"]).copy()
df_manifest = df_manifest.reset_index(drop=True)

arch_counters = {}
graph_ids = []
for _, row in df_manifest.iterrows():
    arch = row["archetype"]
    arch_counters[arch] = arch_counters.get(arch, 0) + 1
    graph_ids.append(f"{arch}_{str(arch_counters[arch]).zfill(2)}")

df_manifest["graph_id"]          = graph_ids
df_manifest["is_representative"]  = df_manifest["graph_id"].str.endswith("_01")

# ─── Step 2: Pre-compute baseline COO once ────────────────────────────
print("Pre-computing baseline COO matrix...")
G_baseline_csr = sp.csr_matrix(
    (W_arr, (sources_arr, targets_arr)),
    shape=(N_GENES, N_GENES)
)
G_baseline_coo = G_baseline_csr.tocoo()

target_baseline_density = 0.0055
lambda_baseline         = N_GENES * target_baseline_density
epsilon                 = 1e-6

# ─── Step 3: Rebuild each graph and save adjacency ────────────────────
print(f"\nRebuilding {len(df_manifest)} champion graphs...")

adj_paths = []

for _, row in tqdm(df_manifest.iterrows(), total=len(df_manifest),
                   desc="Rebuilding cohort"):

    graph_id = row["graph_id"]
    out_path = phase6_dir / f"{graph_id}_adjacency.npz"

    if out_path.exists():
        adj_paths.append(str(out_path))
        continue

    T_norm        = compute_dynamic_topology(G_baseline_coo, row["k_core"], N_GENES)
    gamma_dynamic = row["gamma"] * (row["lambda"] / lambda_baseline)

    numerator    = (W_arr ** row["beta"]) * (1 + row["delta"] * T_norm)
    denominator  = (np.log1p(D_arr) + epsilon) ** gamma_dynamic
    omega_scores = numerator / denominator

    surviving_sources, surviving_targets, surviving_W = dual_pass_pruning(
        omega_scores, W_arr, sources_arr, targets_arr,
        perturbed_nodes, N_GENES, row["lambda"], 3
    )

    adj = sp.csr_matrix(
        (surviving_W, (surviving_sources, surviving_targets)),
        shape=(N_GENES, N_GENES)
    )
    sp.save_npz(str(out_path), adj)
    adj_paths.append(str(out_path))

df_manifest["adjacency_path"] = adj_paths

# ─── Step 4: Write hyphae_manifest.csv ────────────────────────────────
manifest_path = phase6_dir / "hyphae_manifest.csv"
df_manifest.to_csv(manifest_path, index=False)

print(f"\n{'=' * 60}")
print(f"  Phase 6 — HYPHAE Manifest Export")
print(f"{'=' * 60}")
print(f"  Graphs saved  : {len(df_manifest)}")
print(f"  Manifest      : {manifest_path}")
print(f"\n  Archetype breakdown:")
breakdown = (
    df_manifest.groupby("archetype")
    .agg(
        n_graphs       = ("graph_id",    "count"),
        representative = ("graph_id",    "first"),
        best_utopia    = ("utopia_loss", "min"),
        worst_utopia   = ("utopia_loss", "max"),
    )
    .reset_index()
)
print(breakdown.to_string(index=False))
print(f"\n  -> Pass hyphae_manifest.csv to FUNGIv5_HYPHAE.ipynb")